# 04 ? Checkout Friction: The $50 Free Shipping Cliff
### E-Commerce Product Analytics ? Search & Conversion Funnel
**Objective:** Empirically evaluate the relationship between Cart Gross Merchandise Value (GMV), shipping fee surcharges, and Cart-to-Order conversion.

---
### Critical PM Question:
Does the $50 shipping fee cause abandonment, or is it merely confounded by user type or basket size?
We evaluate stratified models across platform, user tier, and item count.


In [ ]:
import sys
import os
sys.path.append('../src')
import duckdb
import pandas as pd
import numpy as np
from exploratory_analysis import get_db_connection, two_proportion_z_test

con = get_db_connection()


## 1. Basket Value Buckets vs Cart-to-Order Conversion


In [ ]:
q_gmv = '''
WITH cart_summary AS (
    SELECT 
        ce.session_id,
        SUM(ce.item_price * ce.quantity) AS cart_gmv,
        MAX(CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END) AS ordered
    FROM cart_events ce
    LEFT JOIN orders o ON ce.session_id = o.session_id
    GROUP BY ce.session_id
)
SELECT 
    CASE 
        WHEN cart_gmv < 25 THEN '1. <$25'
        WHEN cart_gmv < 38 THEN '2. $25-$37.99'
        WHEN cart_gmv < 50 THEN '3. $38-$49.99 (Cliff zone)'
        WHEN cart_gmv < 75 THEN '4. $50-$74.99 (Free ship)'
        WHEN cart_gmv < 100 THEN '5. $75-$99.99'
        ELSE '6. $100+' 
    END AS gmv_bucket,
    COUNT(*) AS total_carts,
    SUM(ordered) AS total_orders,
    ROUND(100.0 * SUM(ordered) / COUNT(*), 2) AS conv_pct
FROM cart_summary
GROUP BY 1
ORDER BY 1
'''
df_gmv = con.execute(q_gmv).df()
print("Checkout Conversion by Cart GMV Tier:")
print(df_gmv.to_string())

# Statistical test on the cliff: $38-$49.99 vs $50-$74.99
c_ord, c_tot = df_gmv.loc[df_gmv['gmv_bucket'].str.contains('38'), ['total_orders', 'total_carts']].values[0]
f_ord, f_tot = df_gmv.loc[df_gmv['gmv_bucket'].str.contains('Free ship'), ['total_orders', 'total_carts']].values[0]

diff, z, p, ci = two_proportion_z_test(f_ord, f_tot, c_ord, c_tot)
print(f"\nFree Shipping Cliff Z-Test: Diff = +{diff*100:.2f} pp, Z = {z:.2f}, p-value = {p:.2e}")


## 2. Stratified Comparison: Controlling for Platform Mix


In [ ]:
q_plat_gmv = '''
WITH cart_summary AS (
    SELECT 
        ce.session_id,
        s.platform,
        SUM(ce.item_price * ce.quantity) AS cart_gmv,
        MAX(CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END) AS ordered
    FROM cart_events ce
    JOIN sessions s ON ce.session_id = s.session_id
    LEFT JOIN orders o ON ce.session_id = o.session_id
    GROUP BY ce.session_id, s.platform
)
SELECT 
    platform,
    CASE 
        WHEN cart_gmv < 38 THEN '1. Sub-$38'
        WHEN cart_gmv < 50 THEN '2. $38-$49.99 (Fee Cliff)'
        WHEN cart_gmv < 75 THEN '3. $50-$74.99 (Free Ship)'
        ELSE '4. $75+' 
    END AS gmv_tier,
    COUNT(*) AS carts,
    SUM(ordered) AS orders,
    ROUND(100.0 * SUM(ordered) / COUNT(*), 2) AS conv_pct
FROM cart_summary
GROUP BY 1, 2
ORDER BY 1, 2
'''
df_pg = con.execute(q_plat_gmv).df()
print("Platform x GMV Tier Matrix:")
print(df_pg.to_string())


## 3. Stratified Comparison: Controlling for User Type (New vs Returning)


In [ ]:
q_usr_gmv = '''
WITH cart_summary AS (
    SELECT 
        ce.session_id,
        u.user_type,
        SUM(ce.item_price * ce.quantity) AS cart_gmv,
        MAX(CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END) AS ordered
    FROM cart_events ce
    JOIN sessions s ON ce.session_id = s.session_id
    JOIN users u ON s.user_id = u.user_id
    LEFT JOIN orders o ON ce.session_id = o.session_id
    GROUP BY ce.session_id, u.user_type
)
SELECT 
    user_type,
    CASE 
        WHEN cart_gmv < 38 THEN '1. Sub-$38'
        WHEN cart_gmv < 50 THEN '2. $38-$49.99 (Fee Cliff)'
        WHEN cart_gmv < 75 THEN '3. $50-$74.99 (Free Ship)'
        ELSE '4. $75+' 
    END AS gmv_tier,
    COUNT(*) AS carts,
    SUM(ordered) AS orders,
    ROUND(100.0 * SUM(ordered) / COUNT(*), 2) AS conv_pct
FROM cart_summary
GROUP BY 1, 2
ORDER BY 1, 2
'''
df_ug = con.execute(q_usr_gmv).df()
print("User Type x GMV Tier Matrix:")
print(df_ug.to_string())


## 4. Visual Evidence


In [ ]:
from IPython.display import Image, display
display(Image(filename='../reports/figures/08_checkout_conversion_vs_basket_value.png'))
display(Image(filename='../reports/figures/10_platform_x_shipping_status.png'))
